In [83]:
1+1

2

In [84]:
a = pd.read_pickle("regressor_11_11_25_w_maagad_feats_results.pkl")

In [48]:
df_names

['T_w_reads_w2_w8.csv',
 'ICS_w_reads_w2_w8.csv',
 'K562_w_reads_w2_w8.csv',
 'U937_w_reads_w2_w8.csv',
 'Fly_w_reads_w2_w8.csv',
 'Shrimp-primary_w_reads_w2_w8.csv',
 'Tomato-Protoplasts_w_reads_w2_w8.csv',
 'Tomato-hairy-roots_eff.csv']

In [268]:
T_df = pd.read_csv(df_names[0])

In [102]:
#T_df['counting'] = T_df['SQA'].apply(lambda x: len(x.split(';')))

In [122]:
#T_df[['g_rna_info','SQA','target_seq']]

In [269]:
# Function to add replicate suffixes
def explode_sqa(df):
    new_rows = []
    for _, row in df.iterrows():
        # print(row['g_rna_info'])
        sqa_values = row['SQA'].split(';')  # split by ';'
        for i, sqa in enumerate(sqa_values, start=1):
            suffix = f"_{i}st_rep" if i == 1 else f"_{i}nd_rep" if i == 2 else f"_{i}rd_rep" if i == 3 else f"_{i}th_rep"
            new_row = row.copy()
            new_row['SQA'] = sqa
            new_row['g_rna_info'] = f"{row['g_rna_info']}{suffix}"
            new_rows.append(new_row)
    return pd.DataFrame(new_rows)

# Create new dataframe
new_T_df = explode_sqa(T_df)
new_T_df = new_T_df[['g_rna_info','SQA','target_seq']].reset_index(drop=True)

In [276]:
import pandas as pd
import os

def assign_sqa_files(df, folder_path):
    """
    For each SQA value in df, find exactly two files in folder_path
    containing the SQA name. Assign them to new columns SQA_R1 and SQA_R2.
    
    Args:
        df (pd.DataFrame): Must have a column 'SQA'.
        folder_path (str): Path to folder containing files.
        
    Returns:
        pd.DataFrame: df with added columns 'SQA_R1' and 'SQA_R2'.
    """
    # Get list of all files in folder
    all_files = os.listdir(folder_path)
    
    # Initialize new columns
    df['SQA_R1'] = None
    df['SQA_R2'] = None
    
    for idx in df.index:
        sqa = df.loc[idx,'SQA'].replace('#','')
        # Find files containing the SQA string
        # print(idx,sqa)
        matching_files = [f for f in all_files if sqa+'_' in f]
        # if '0001191' in sqa:
            # print(sqa,matching_files[0], matching_files[1])
        # matching_files = matching_files[:2]
        if len(matching_files) != 2:
            # print(sqa,matching_files)
            # matching_files = [f for f in all_files if f'{sqa}.fastq.gz' == f]
            # if len(matching_files) != 2:
            print('BAD',sqa,matching_files)
            raise ValueError(f"SQA '{sqa}' does not have exactly 2 matching files in '{folder_path}'. Found: {matching_files}")
        
        df.loc[idx, 'SQA_R1'] = matching_files[0]
        df.loc[idx, 'SQA_R2'] = matching_files[1]
        # if '0001191' in sqa: 
        #     print('here',idx)
        #     print(df.loc[idx,'SQA_R1'])
            # print(df.at[idx,'SQA_R2'])
        all_files = [x for x in all_files if x not in matching_files]
        # print(matching_files)
    print(all_files)
    return df.drop(columns=['SQA'])


In [277]:
folder_path = "/tamir2/shaicohen1/CRISPR_MAAGAD/all_raw_data/T/"
new_T_df = assign_sqa_files(new_T_df, folder_path)

['.ipynb_checkpoints']


In [278]:
new_T_df

,g_rna_info,target_seq,SQA_R1,SQA_R2
0,1_244860352_244860372_+_1st_rep,TTTGCCTTTTGACACACCATAGG,SQA0011372_g151_S1_R1.fastq.gz,SQA0011372_g151_S1_R2.fastq.gz
1,3_187733542_187733562_-_1st_rep,AGCCCATAAAACGGTCCTCATGG,SQA0011394_g173_S11_R1.fastq.gz,SQA0011394_g173_S11_R2.fastq.gz
2,16_56351464_56351484_+_1st_rep,CTCAACAAGAAAGATCTCTTTGG,SQA0007676_g142_S33_R1.fastq.gz,SQA0007676_g142_S33_R2.fastq.gz
3,18_33745555_33745575_-_1st_rep,GTTCTGCTGGAAGCTACACGAGG,SQA0007678_g144_S35_R1.fastq.gz,SQA0007678_g144_S35_R2.fastq.gz
4,6_106105227_106105247_-_1st_rep,CAGGGGACACCGTATTCCCAGGG,SQA0001191_31B_S56_R1.fastq.gz,SQA0001191_31B_S56_R2.fastq.gz
...,...,...,...,...
205,18_33746019_33746039_-_1st_rep,GTGGTTTCCATGGTAACTGGAGG,SQA0007679_g145_S36_R1.fastq.gz,SQA0007679_g145_S36_R2.fastq.gz
206,2_218134970_218134990_-_1st_rep,CAGGCTCAGCAGGAATACCAGGG,SQA0007669_g135_S6_R1.fastq.gz,SQA0007669_g135_S6_R2.fastq.gz
207,2_218134970_218134990_-_2nd_rep,CAGGCTCAGCAGGAATACCAGGG,SQA0007669_g135_S29_R1.fastq.gz,SQA0007669_g135_S29_R2.fastq.gz
208,2_111150097_111150117_-_1st_rep,TCCAATACGCCGCAACTCTTGGG,SQA0002491_34A_S29_R1.fastq.gz,SQA0002491_34A_S29_R2.fastq.gz


In [284]:
import pandas as pd

# Example input dataframe
# df = pd.read_csv("your_input.csv")  # or however you load it
# df columns: g_rna_info, SQA_R1, SQA_R2, target_seq

def generate_sra_tsv(df, output_file="tcell_sra_metadata.tsv"):
    """
    Generate SRA metadata TSV from gRNA amplicon dataframe.
    
    Parameters
    ----------
    df : pd.DataFrame
        Must contain columns: g_rna_info, SQA_R1, SQA_R2, target_seq
    output_file : str
        Path to save the TSV
    """
    
    # Prepare output dataframe
    sra_columns = [
        "sample_name", "library_ID", "title", "library_strategy",
        "library_source", "library_selection", "library_layout",
        "platform", "instrument_model", "design_description",
        "filetype", "filename", "filename2", "filename3", "filename4",
        "assembly", "fasta_file"
    ]
    
    output_rows = []
    
    # Keep track of replicate counts per gRNA
    replicate_count = {}
    
    for idx, row in df.iterrows():
        g_rna = row['g_rna_info']
        
        # Determine replicate number
        rep_num = int(g_rna.split('_')[-2][:-2])
        rep_suffix = '_'.join(g_rna.split('_')[-2:])
        
        # print(rep_suffix)
        library_id = f"{row['target_seq']}_{rep_suffix}"
        title = f"T cells CRISPR amplicon {'_'.join(g_rna.split('_')[:-2])}" + (f" replicate {rep_num}" if rep_num > 1 else "")
        
        # Build design description
        design_description = (
            f"Human T cells isolated from blood, "
            f"CRISPR–Cas9 RNP electroporation, amplicon library for gRNA targeting "
            f"{g_rna} (target sequence: {row['target_seq']}) prepared using two-step PCR"
        )
        
        output_rows.append({
            "sample_name": "Human_Tcell_Donor01",
            "library_ID": library_id,
            "title": title,
            "library_strategy": "AMPLICON",
            "library_source": "GENOMIC",
            "library_selection": "PCR",
            "library_layout": "PAIRED",
            "platform": "ILLUMINA",
            "instrument_model": "Illumina MiSeq",
            "design_description": design_description,
            "filetype": "fastq",
            "filename": row['SQA_R1'],
            "filename2": row['SQA_R2'],
            "filename3": "",
            "filename4": "",
            "assembly": "",
            "fasta_file": ""
        })
    
    out_df = pd.DataFrame(output_rows, columns=sra_columns)
    return out_df
    # # Save as TSV
    # out_df.to_csv(output_file, sep="\t", index=False)
    # print(f"SRA metadata TSV saved to: {output_file}")

# Example usage:
final_T_df =generate_sra_tsv(new_T_df, "tcell_sra_metadata.tsv")


In [285]:
final_T_df.to_csv('/tamir2/shaicohen1/CRISPR_MAAGAD/all_raw_data/T_metadata.tsv', sep="\t", index=False)

In [238]:
new_T_df[new_T_df['SQA'].str.contains('SQA0001191')]

,g_rna_info,SQA,target_seq,SQA_R1,SQA_R2
4,6_106105227_106105247_-_1st_rep,SQA0001191_31B_S56,CAGGGGACACCGTATTCCCAGGG,SQA0001191_31B_S93_R1.fastq.gz,SQA0001191_31B_S93_R2.fastq.gz
4,6_106105227_106105247_-_2nd_rep,SQA0001191_31B_S93,CAGGGGACACCGTATTCCCAGGG,SQA0001191_31B_S93_R1.fastq.gz,SQA0001191_31B_S93_R2.fastq.gz


In [145]:
files = os.listdir(folder_path)

In [214]:
[x for x in files if 'SQA0001191' in x] 

['SQA0001191_31B_S56_R1.fastq.gz',
 'SQA0001191_31B_S56_R2.fastq.gz',
 'SQA0001191_31B_S93_R1.fastq.gz',
 'SQA0001191_31B_S93_R2.fastq.gz']

In [156]:
new_T_df[new_T_df['SQA'].str.contains('SQA0007656')]

,g_rna_info,SQA,target_seq,SQA_R1,SQA_R2
62,10_95711940_95711960_+_1st_rep,SQA0007656_g122_S2,TCTTCTGGAAGCTGCAATGAAGG,None,None
62,10_95711940_95711960_+_2nd_rep,SQA0007656_g122_S20,TCTTCTGGAAGCTGCAATGAAGG,None,None


In [43]:
all_df = pd.DataFrame({})
x
for x in df_names:
    df = pd.read_csv(x)
    break
    x = x.split('_')[0]
    x = x.replace('Shrimp-primary','Prawn').replace('Tomato-Protoplasts','Tomato Leaf').replace('Tomato-hairy-roots','Tomato Hairy Roots').replace('ICS','PLX')
    if x in ['PLX','K562','U937','T']:
        organism = 'Human'
        tissue = x
    elif 'Tomato' in x:
        organism = 'Tomato'
        tissue = x.split(' ')[1]
    else:
        organism = x
        if x=='Prawn': 
            tissue = 'Primary Cell Culture'
        if x=='Fly': 
            tissue = 'Embryo'
    
    editing_eff_col = 'editing_efficinecy_w8_no_subs_18_9_25'
    if x == 'Tomato Hairy Roots':
        editing_eff_col = 'editing_efficiency'
        # df_tomato = 
    df = df[~df[editing_eff_col].isna()]
    if 'sqa' in df.columns: 
        print(x, 'SQA')
    else:
        print(x,'no')
    # for x in df.
    # df['organism'] = organism
    # df['tissue'] = tissue
    # all_df = pd.concat([all_df,df],axis=0)
print('done')

done


In [46]:
df.columns.to_list()

['Unnamed: 0.2',
 'Unnamed: 0.1',
 'Unnamed: 0',
 'g_rna_info',
 'RNA_minimum_free_energy',
 'RNA_ratio_of_paired_bases',
 'RNA_long_tail_len',
 'RNA_short_tail_len',
 'RNA_sum_len_of_tails',
 'RNA_G_C_paired_ratio',
 'RNA_A_U_paired_ratio',
 'conservation_mean',
 'conservation_extended_mean',
 'conservation_1',
 'conservation_2',
 'conservation_3',
 'conservation_4',
 'conservation_5',
 'conservation_6',
 'conservation_7',
 'conservation_8',
 'conservation_9',
 'conservation_10',
 'conservation_11',
 'conservation_12',
 'conservation_13',
 'conservation_14',
 'conservation_15',
 'conservation_16',
 'conservation_17',
 'conservation_18',
 'conservation_19',
 'conservation_20',
 'conservation_21',
 'conservation_22',
 'conservation_23',
 'DNA_fold',
 'expression_max_win0_expr_HEK293',
 'expression_max_win0_expr_HSPC',
 'expression_max_win0_expr_iPSC',
 'expression_max_win0_expr_k562',
 'expression_max_win0_expr_PLX',
 'expression_max_win0_expr_T',
 'expression_max_win0_expr_THP',
 'expr

In [37]:
all_df.to_csv('/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/final_feats/Samples_in_all_data.csv',index=False)

In [39]:
df.SiteName

0       amp-1-T1
1       amp-1-T2
2       amp-2-T1
3       amp-3-T1
4       amp-4-T1
         ...    
99     amp-61-T1
100    amp-62-T1
101    amp-62-T2
102    amp-63-T1
103    amp-63-T2
Name: SiteName, Length: 104, dtype: object

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import make_scorer
from sklearn.model_selection import KFold
from statsmodels.stats.multitest import multipletests
from scipy.stats import mannwhitneyu, wilcoxon, spearmanr
import os
import seaborn as sns
import matplotlib.pyplot as plt
import itertools
import xgboost as xgb
from xgboost import XGBRegressor
from collections import Counter
import traceback

pd.set_option('display.precision', 3)

In [2]:
df_names = ['T_w_reads_w2_w8.csv',
            'ICS_w_reads_w2_w8.csv',
            'K562_w_reads_w2_w8.csv',
            'U937_w_reads_w2_w8.csv',
            'Fly_w_reads_w2_w8.csv',
            'Shrimp-primary_w_reads_w2_w8.csv',
            'Tomato-Protoplasts_w_reads_w2_w8.csv',
            'Tomato-hairy-roots_eff.csv']
# df_names = ['T_w_reads_w2_w8.csv',
#             'ICS_w_reads_w2_w8.csv',
#             'K562_w_reads_w2_w8.csv',
#             'U937_w_reads_w2_w8.csv',
#             'Fly_w_reads_w2_w8.csv',
#             'Shrimp-primary_w_reads_w2_w8.csv',
#             'Shrimp-embryos_w_reads_w2_w8.csv',
#             'Tomato-Protoplasts_w_reads_w2_w8.csv',
#             'Tomato-hairy-roots_eff.csv']
# df_names = ['DeepHF_final.csv']
concised_names=[]
for df_name in df_names:
    concised_names.append(df_name.split('_')[0])

In [82]:
def make_rep_feats(df,edit_eff_name):
    feats_to_keep = []
    cai_and_chimera_cols = [f'{name}_{f}{m}_{win}'
        for win in [0] 
        for name,f in zip(['codon','aa','CAI','chimera'],['freqs_','freqs_','','']) 
        for m in ['avg']]
    # tad = ['tad_density','tad_angle','tad_interactions']
    # feats_to_keep.extend(tad)
    # orenstein = ['RRBS','H3K4me3','Dnase','CTCF']
    # feats_to_keep.extend(orenstein)
    isana = ['g_DNADNA','g_RNADNA','guideEne','guide&scafEne','clause','isBasePairs','Head1','Head2','Head3','1&2&3']
    feats_to_keep.extend(cai_and_chimera_cols)
    feats_to_keep.extend(isana)
    feats_to_keep.extend([x for x in df.columns.to_list() if 'DNAshape' in x 
                  and 'mean' in x and 'ext' not in x])
    feats_to_keep.extend([x for x in df.columns.to_list() if 'enthalpy' in x
                      and 'mean' in x and 'ext' not in x])
    LCC_feats = [x for x in df.columns.to_list() if 'LCC' in x]
    df.loc[:,'LCC_rep'] = df[LCC_feats].mean(axis=1)
    feats_to_keep.append('LCC_rep')
    feats_to_keep.extend([x for x in df.columns.to_list() if 'methylation' in x
                      and 'mean' in x and 'ext' not in x])
    sites_feats = [x for x in df.columns.to_list() if 'sites_count' in x
     and (x[-2:] in '-0-5' or x[-2:] in 'd0d2d5')
     and 'orf' not in x and 'full' not in x and 'global' not in x
     and(('ngg_pam_1000_' in x or 'count_1000_' in x)
     or ('ngg_pam_10000_' in x or 'count_10000_' in x)
     or ('ngg_pam_100000_' in x or 'count_100000_' in x)
     or ('ngg_pam_100000_' in x or 'count_100000_' in x))]
    sites_feats.extend(['sites_count_ngg_pam_genome_-10_d0','sites_count_ngg_pam_genome_-10_d2','sites_count_ngg_pam_genome_-10_d5'])
    feats_to_keep.extend(sites_feats)
    df.loc[:,'k5_Q1_rep'] = df[[x for x in df.columns.to_list() if 'spacers' in x
    and 'Q1' in x and 'k5' in x and 'GCvsAT' not in x]].mean(axis=1)
    df.loc[:,'k5_Q4_rep'] = df[[x for x in df.columns.to_list() if 'spacers' in x
    and 'Q4' in x and 'k5' in x and 'GCvsAT' not in x]].mean(axis=1)
    feats_to_keep.extend(['k5_Q1_rep','k5_Q4_rep'])
    feats_to_keep.append(edit_eff_name)
    return df[feats_to_keep]

In [ ]:
def set_model(model_name):
    if model_name=='linear':
        model = LinearRegression()
    elif model_name=='rand_forest':
        model = RandomForestRegressor(
                n_estimators=300,
                max_depth=3,
                min_samples_split=3,
                min_samples_leaf=2,
                max_features=None,
                random_state=42
            )
    elif model_name == 'xgb':
          model =  XGBRegressor(
                n_estimators=200,        # more trees but each contributes little
                learning_rate=0.05,      # slow learning to prevent overfitting
                max_depth=2,             # shallow trees
                subsample=0.8,           # add randomness
                colsample_bytree=0.8,
                reg_lambda=1.0,          # L2 regularization
                reg_alpha=0.0,           # L1 regularization (set >0 if you want stronger shrinkage)
                random_state=0,
                n_jobs=-1
            )
    return model

In [9]:
# Define custom Spearman scorer
def spearman_scorer(y_true, y_pred):
    r = spearmanr(y_true, y_pred)[0]
    return -1e6 if np.isnan(r) else r
spearman_scorer_skl = make_scorer(spearman_scorer, greater_is_better=True)
def adjusted_spearman_scorer(estimator, X, y):
    # Predict using current feature subset
    y_pred = estimator.predict(X)
    
    # Compute Spearman correlation
    r = spearmanr(y, y_pred)[0]
    if np.isnan(r):
        return -1e6

    # Adjust for number of features
    n = len(y)
    p = X.shape[1]
    r2 = r ** 2
    adj_r2 = 1 - (1 - r2) * (n - 1) / max(1, (n - p - 1))

    # Return sign of r (since r2 loses sign)
    adj_r = np.sign(r) * np.sqrt(max(0, adj_r2))
    return adj_r

# sklearn scorer object
adjusted_spearman_scorer_skl = make_scorer(adjusted_spearman_scorer, greater_is_better=True)
def run_forward_feature_selection(X_train, y_train, model_name, random_state=0):
    """
    Runs forward feature selection with CV using Spearman correlation.
    Works even if y_train is a numpy array.
    Returns: selected feature names (list)
    """

    # Ensure y_train is pandas Series for compatibility
    if isinstance(y_train, np.ndarray):
        y_train = pd.Series(y_train, index=X_train.index)

    if X_train.shape[1]!=1:
        # Define CV
        kf = KFold(n_splits=5, shuffle=True, random_state=random_state)
    
        # Define Sequential Feature Selector
        sfs = SequentialFeatureSelector(
            estimator=set_model(model_name),
            n_features_to_select="auto",  # auto-stops when CV score stops improving
            direction='forward',
            scoring=adjusted_spearman_scorer_skl,
            cv=kf,
            n_jobs=-1
        )
    
        # Fit selector
        sfs.fit(X_train, y_train)
    
        # Get selected feature names
        selected_feats = list(X_train.columns[sfs.get_support()])
    else:
        selected_feats = X_train.columns.to_list()
    
    final_model = set_model(model_name)
    final_model.fit(X_train[selected_feats], y_train)
    return final_model, selected_feats

In [10]:
#####
def forward_feature_selection_with_cv(model_name, X_in_func, y_in_func, n_splits=5, random_state=42, verbose=False):
    """
    Forward feature selection using internal cross-validation.
    Reduces overfitting to a single validation split.
    """
    if hasattr(X_in_func, "columns"):
        feat_names = list(X_in_func.columns)
    else:
        feat_names = [f"feat_{i}" for i in range(X_in_func.shape[1])]
        X_in_func = pd.DataFrame(X_in_func, columns=feat_names)

    if not isinstance(y_in_func, pd.Series):
        y_in_func = pd.Series(y_in_func).reset_index(drop=True)
    
    model = set_model(model_name)
    remaining_feats = feat_names.copy()
    selected_feats = []
    best_score = -np.inf
    rng = np.random.RandomState(random_state)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    # print(remaining_feats)
    step = 1
    if len(feat_names)==1:
        selected_feats=feat_names
    else:
        while remaining_feats:
            scores = []
            if verbose:
                print(f"\n=== Step {step}: testing additions to {selected_feats or '[]'} ===")
    
            for feat in remaining_feats:
                feats_to_try = selected_feats + [feat]
                fold_scores = []
    
                # Cross-validation for this feature subset
                for train_idx, val_idx in kf.split(X_in_func):
                    X_train, X_val = X_in_func.iloc[train_idx][feats_to_try], X_in_func.iloc[val_idx][feats_to_try]
                    y_train, y_val = y_in_func.iloc[train_idx], y_in_func.iloc[val_idx]
                    model.fit(X_train, y_train)
                    preds = model.predict(X_val)
                    fold_scores.append(spearmanr(y_val, preds)[0])
    
                mean_score = np.nanmedian(fold_scores)
                scores.append((feat, mean_score))
                if verbose:
                    print(f"  Try {feats_to_try}: median CV Spearman = {mean_score:.3f}")
    
            best_feat, best_feat_score = max(scores, key=lambda x: x[1])
            if verbose:
                print(f"Best new feat: {best_feat} (median CV Spearman {best_feat_score:.3f})")
    
            # Stop if no improvement
            if best_feat_score <= best_score:
                if verbose:
                    print("No improvement — stopping.")
                break
    
            selected_feats.append(best_feat)
            remaining_feats.remove(best_feat)
            best_score = best_feat_score
            step += 1

    if verbose:
        print(f"\n✅ Final selected features: {selected_feats}")
        print(f"✅ Best median CV Spearman: {best_score:.3f}")

    # Fit final model on *all training data* using selected features
    final_model = set_model(model_name)
    final_model.fit(X_in_func[selected_feats], y_in_func)

    return final_model, selected_feats


In [89]:
import pickle

file_path = '/tamir2/shaicohen1/CRISPR_MAAGAD/CRISPR_review/code/final_feats/13_11_25_results/xgb/xgb_Tomato-hairy-roots_13_11_25_spearman_results.pkl'

with open(file_path, 'rb') as f:
    data = pickle.load(f)

print(type(data))
# Optional: preview keys or content
if isinstance(data, dict):
    print(data.keys())
else:
    print(data)


<class 'dict'>
dict_keys(['CRISPRedict_eff', 'DeepCRISPR_eff', 'uCRISPR_eff', 'SPROUT_eff', 'CRISPRedict_eff, DeepCRISPR_eff', 'CRISPRedict_eff, uCRISPR_eff', 'CRISPRedict_eff, SPROUT_eff', 'DeepCRISPR_eff, uCRISPR_eff', 'DeepCRISPR_eff, SPROUT_eff', 'uCRISPR_eff, SPROUT_eff', 'CRISPRedict_eff, DeepCRISPR_eff, uCRISPR_eff', 'CRISPRedict_eff, DeepCRISPR_eff, SPROUT_eff', 'CRISPRedict_eff, uCRISPR_eff, SPROUT_eff', 'DeepCRISPR_eff, uCRISPR_eff, SPROUT_eff', 'CRISPRedict_eff, DeepCRISPR_eff, uCRISPR_eff, SPROUT_eff'])


In [ ]:
aa

In [20]:
def reg_analysis(csv_path, editing_eff_col, model_name='linear', run_combinations=False,
                 n_iters=20,base_out_dir='.',verbose=False):
    """
    Runs regression experiments across subsets of features, including:
      - ALL
      - single features
      - optional combinations (if run_combinations=True)
      - all leave-one-out sets (always)
    Saves plots and returns (results_dict, one_row_summary_df).
    """
    _df = pd.read_csv(csv_path)
    if csv_path == 'Tomato-hairy-roots_eff.csv':
        editing_eff_col = 'editing_efficiency'
    _df = _df[~_df[editing_eff_col].isna()]

    # feature list (original names include '_eff')
    features = ['CRISPRedict_eff', 'DeepCRISPR_eff', 'uCRISPR_eff', 'SPROUT_eff']
    all_feats_tuple = tuple(features)
    missing = [c for c in features + [editing_eff_col] if c not in _df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    dataset_name = csv_path.split('_')[0]
    X_full = _df[features].values
    y_all = _df[editing_eff_col].values.astype(float)

    test_size = 0.2
    if csv_path == 'Shrimp-embryos_w_reads_w2_w8.csv':
        test_size = 0.5

    base_seed = 12345
    train_rand_seed = 194
    # Output directories
    
    dataset_dir = os.path.join(base_out_dir, dataset_name)
    os.makedirs(dataset_dir, exist_ok=True)
    
    def clean_feat(f):
        return f.replace("_eff", "")

    feats_list = [    ['CRISPRedict_eff'],    ['DeepCRISPR_eff'],    ['uCRISPR_eff'],    ['SPROUT_eff'],    ['CRISPRedict_eff', 'DeepCRISPR_eff'],    ['CRISPRedict_eff', 'uCRISPR_eff'],    ['CRISPRedict_eff', 'SPROUT_eff'],    ['DeepCRISPR_eff', 'uCRISPR_eff'],    ['DeepCRISPR_eff', 'SPROUT_eff'],    ['uCRISPR_eff', 'SPROUT_eff'],    ['CRISPRedict_eff', 'DeepCRISPR_eff', 'uCRISPR_eff'],    ['CRISPRedict_eff', 'DeepCRISPR_eff', 'SPROUT_eff'],    ['CRISPRedict_eff', 'uCRISPR_eff', 'SPROUT_eff'],    ['DeepCRISPR_eff', 'uCRISPR_eff', 'SPROUT_eff'],    ['CRISPRedict_eff', 'DeepCRISPR_eff', 'uCRISPR_eff', 'SPROUT_eff']]
    results = {}
    def display_name_from_tuple(feat_tuple):
        feat_set = set(feat_tuple)
        if feat_set == set(all_feats_tuple):
            return "ALL"
        missing = set(all_feats_tuple) - feat_set
        if len(missing) == 1:
            return "leave_" + clean_feat(next(iter(missing)))
        return "_".join([clean_feat(f) for f in feat_tuple])

    results = {}
    for selected_feats in feats_list:
        feat_tuple= tuple(selected_feats)
        disp_name = display_name_from_tuple(feat_tuple)
        print(selected_feats)
        X = _df[selected_feats]
        all_selected_feats=[]
        all_RAND_selected_feats=[]
        spearman_real, spearman_perm = [], []     
        for i in range(n_iters):
            idx = np.arange(len(y_all))
            rng_i = np.random.default_rng(base_seed + i)
            rng_i.shuffle(idx)
            n_test = int(np.floor(test_size * len(idx)))
            test_idx, train_idx = idx[:n_test], idx[n_test:]

            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y_all[train_idx], y_all[test_idx]
            # real model
            X_train = X_train[selected_feats]
            X_test = X_test[selected_feats]
            all_selected_feats.append(selected_feats)
            model = set_model(model_name)
            model.fit(X_train,y_train)
            y_pred = model.predict(X_test[selected_feats])
            rho_real, _ = spearmanr(y_test, y_pred)
            spearman_real.append(rho_real)

            # permuted model
            y_train_perm = y_train.copy()
            rng_i = np.random.default_rng(train_rand_seed + i)
            rng_i.shuffle(y_train_perm)
            model_perm = set_model(model_name)
            model_perm.fit(X_train,y_train_perm)
            
            perm_selected_feats = selected_feats
            all_RAND_selected_feats.append(perm_selected_feats)
            y_pred_perm = model_perm.predict(X_test[perm_selected_feats])
            rho_perm, _ = spearmanr(y_test, y_pred_perm)
            spearman_perm.append(rho_perm)
            # if i%25==0 and n_iters>=50: print(i)

        spearman_real = np.array(spearman_real)
        spearman_perm = np.array(spearman_perm)
        gain = spearman_real - spearman_perm

        results[disp_name] = {
            "starting_feats": feat_tuple,
            "selected_feats": all_selected_feats,
            "perm_selected_feats": all_RAND_selected_feats,
            "real": spearman_real,
            "perm": spearman_perm,
            "gain": gain,
            "median_real": np.nanmedian(spearman_real),
            "median_perm": np.nanmedian(spearman_perm)
        }

        # Save per-model plot (real vs permuted) with p-value in legend
        res = stats.wilcoxon(gain, alternative='greater', zero_method='wilcox')
        p_val = res.pvalue
        # _, p_val = mannwhitneyu(spearman_real, spearman_perm, alternative='greater')
        p_str = f"p={p_val:.2e}"

        plt.figure(figsize=(8,5))
        num_bins = min(20, len(y_all))
        plt.hist(spearman_real, bins=num_bins, alpha=0.6, label='Real')
        plt.hist(spearman_perm, bins=num_bins, alpha=0.6, label='Permuted')
        plt.axvline(np.nanmedian(spearman_real), color='blue', linestyle='dashed', linewidth=2,
                    label=f"Real median={np.nanmedian(spearman_real):.2f}")
        plt.axvline(np.nanmedian(spearman_perm), color='orange', linestyle='dashed', linewidth=2,
                    label=f"Perm median={np.nanmedian(spearman_perm):.2f}")
        plt.title(f"{dataset_name}: {disp_name}")
        plt.xlabel("Spearman rho")
        plt.ylabel("Count")
        plt.legend(title=p_str)  # show p in legend
        plt.tight_layout()

        per_model_path = os.path.join(dataset_dir, f"{dataset_name}_{disp_name}.svg")
        plt.savefig(per_model_path, format='svg')
        plt.close()

    # ALL vs subset plots
    baseline_real = results["ALL"]["real"]
    baseline_gain = results["ALL"]["gain"]

    for name, vals in results.items():
        if name == "ALL":
            continue
        
        # _, p_val = mannwhitneyu(baseline_real, vals["real"], alternative='greater')
        if (baseline_real - vals["real"]).all() == 0:
            p_val = 1
        else:
            res = stats.wilcoxon(baseline_real-vals["real"], alternative='greater', zero_method='wilcox')
            p_val = res.pvalue
        p_str = f"p={p_val:.2e}"

        plt.figure(figsize=(8,5))
        num_bins = min(20, len(y_all))
        plt.hist(baseline_real, bins=num_bins, alpha=0.6, label='ALL (real)')
        plt.hist(vals["real"], bins=num_bins, alpha=0.6, label=f"{name} (real)")
        plt.axvline(np.nanmedian(baseline_real), color='blue', linestyle='dashed', linewidth=2,
                    label=f"ALL median={np.nanmedian(baseline_real):.2f}")
        plt.axvline(vals["median_real"], color='orange', linestyle='dashed', linewidth=2,
                    label=f"{name} median={vals['median_real']:.2f}")
        plt.title(f"{dataset_name}: ALL vs {name} (real rhos)")
        plt.xlabel("Spearman rho")
        plt.ylabel("Count")
        plt.legend(title=p_str)  # show p in legend
        plt.tight_layout()

        all_vs_path = os.path.join(dataset_dir, f"{dataset_name}_ALL_vs_{name}.svg")
        plt.savefig(all_vs_path, format='svg')
        plt.close()

    # Build single-row summary
    summary_row = {"name": dataset_name}
    summary_row["ALL_rho"] = results["ALL"]["median_real"]

    # Group 1: ALL vs subsets (real)
    for name, vals in results.items():
        if name == "ALL":
            continue
        # _, p = mannwhitneyu(baseline_real, vals["real"], alternative='greater')
        if (baseline_real - vals["real"]).all() == 0: 
            p=1
        else:
            res = stats.wilcoxon(baseline_real-vals["real"], alternative='greater', zero_method='wilcox')
            p = res.pvalue
        if name.startswith("leave_"):
            feat = name[len("leave_"):]
            summary_row[f"leave_{feat}_p_val"] = p
        else:
            summary_row[f"ALL_v_{name}_p_val"] = p

    # Group 2: RAND vs real (self)
    for name, vals in results.items():
        # _, p = mannwhitneyu(vals["real"], vals["perm"], alternative='greater')
        if (vals["real"]- vals["perm"]).all() == 0: 
            p=1
        else:
            res = stats.wilcoxon(vals["real"]-vals["perm"], alternative='greater', zero_method='wilcox')
            p = res.pvalue
        if name == "ALL":
            summary_row["RAND_ALL_p_val"] = p
        elif name.startswith("leave_"):
            feat = name[len("leave_"):]
            summary_row[f"leave_{feat}_rand_p_val"] = p
        else:
            summary_row[f"{name}_rand_p_val"] = p

    # Group 4: median rhos
    for name, vals in results.items():
        summary_row[f"{name}_rho"] = vals["median_real"]
        summary_row[f"RAND_{name}_rho"] = vals["median_perm"]

    summary_df = pd.DataFrame([summary_row])

    out_path = os.path.join(base_out_dir, f"{dataset_name}_{editing_eff_col}_summary.csv")
    os.makedirs(base_out_dir, exist_ok=True)
    summary_df.to_csv(out_path, index=False)
    print(f"Saved summary to {out_path}")

    return results, summary_df


In [ ]:
# for the_model_name in ['linear','xgb','rand_forest']:
from time import time
a=time()
for the_model_name in ['linear','rand_forest','xgb']:
    with open("error_log.txt", "a") as f:
        f.write(f'{the_model_name}')
    print(the_model_name)
    try:
        final_summary_df = pd.DataFrame()   # start empty
        final_results_df = pd.DataFrame()   # start empty
        raw_cols = ['selected_feats',	'RAND_selected_feats','median_real'	,'median_perm']
        folder= f"./21_10_25_results/{the_model_name}/"
        for df_name in df_names:
            print(df_name)
            concised_name = df_name.split("_")[0]
            results, summary_df =  reg_analysis(
                df_name,
                'editing_efficinecy_w8_no_subs_18_9_25',
                model_name= the_model_name,#rand_forest
                base_out_dir=folder,
                n_iters=100,
                verbose=False
            )
            results_df = pd.DataFrame.from_dict(results, orient='index')
            results_df = results_df.rename(columns={
                'perm_selected_feats': 'RAND_selected_feats'
            })
            # results_df = results_df[raw_cols]
            # results_df.index = [f'{concised_name}_'+x for x in results_df.index]
            results_df = results_df.astype(object)
            summary_df = summary_df
            for idx in results_df.index:
                cols = [idx+'_'+x for x in raw_cols]
                for raw_c,c in zip(raw_cols,cols):
                    if 'selected_feats' in raw_c:
                        normalized_data = [tuple(sorted(lst)) for lst in results_df.loc[idx,raw_c]]
                        frequency_counts = Counter(normalized_data)
                        to_save =  str(frequency_counts)
                    else:
                        to_save = results_df.loc[idx,raw_c]
                    summary_df.loc[0,c] = to_save
        
                
            # summary_df = pd.concat([summary_df,results_df], axis=1 )
            final_results_df = pd.concat([final_results_df, results_df], ignore_index=False)
            
            final_summary_df = pd.concat([final_summary_df, summary_df], ignore_index=True)
            # save intermediate results each iteration (optional, good for safety)
            final_summary_df.to_csv(folder+f'/w8_no_subs_{the_model_name}_summary.csv', index=False)
            final_results_df.to_csv(folder+f'/w8_no_subs_{the_model_name}_results.csv', index=False)
            # break
        # break
    except Exception as e:
        # Open (or create) a file to save the error
        with open("error_log.txt", "a") as f:
            f.write(f"{the_model_name} {concised_name}:\n\n")
            f.write("An error occurred:\n\n")
            # Write the error message
            f.write(str(e) + "\n\n")
            # Write the full traceback for debugging
            f.write("Full traceback:\n")
            f.write(traceback.format_exc())
        print(the_model_name,concised_name,'error')
        continue
    print("done")
    # break
print("all done")
print(time()-a)
with open("error_log.txt", "a") as f:
    f.write(f'{time()-a}')

linear
T_w_reads_w2_w8.csv
['CRISPRedict_eff']
['DeepCRISPR_eff']
['uCRISPR_eff']
['SPROUT_eff']
['CRISPRedict_eff', 'DeepCRISPR_eff']
['CRISPRedict_eff', 'uCRISPR_eff']
['CRISPRedict_eff', 'SPROUT_eff']
['DeepCRISPR_eff', 'uCRISPR_eff']
['DeepCRISPR_eff', 'SPROUT_eff']
['uCRISPR_eff', 'SPROUT_eff']
['CRISPRedict_eff', 'DeepCRISPR_eff', 'uCRISPR_eff']
['CRISPRedict_eff', 'DeepCRISPR_eff', 'SPROUT_eff']
['CRISPRedict_eff', 'uCRISPR_eff', 'SPROUT_eff']
['DeepCRISPR_eff', 'uCRISPR_eff', 'SPROUT_eff']
['CRISPRedict_eff', 'DeepCRISPR_eff', 'uCRISPR_eff', 'SPROUT_eff']
Saved summary to ./21_10_25_results/linear/T_editing_efficinecy_w8_no_subs_18_9_25_summary.csv
ICS_w_reads_w2_w8.csv
['CRISPRedict_eff']
['DeepCRISPR_eff']
['uCRISPR_eff']
['SPROUT_eff']
['CRISPRedict_eff', 'DeepCRISPR_eff']
['CRISPRedict_eff', 'uCRISPR_eff']
['CRISPRedict_eff', 'SPROUT_eff']
['DeepCRISPR_eff', 'uCRISPR_eff']


In [74]:
all_models_df = pd.DataFrame(columns=final_summary_df.name.to_list(), index= ['rand_forest','xgb','linear'])
for the_model_name in ['rand_forest','xgb','linear']:
    folder = f"./21_10_25_results/{the_model_name}/"
    final_summary_df = pd.read_csv(folder+f'/w8_no_subs_{the_model_name}_summary.csv')
    a_df = final_summary_df[[x for x in final_summary_df.columns if x.endswith('real')]]
    max_cols = a_df.idxmax(axis=1)
    max_vals = a_df.max(axis=1)
    all_models_df.loc[the_model_name] = [f'{x[:-12]} {y:.2f}' for x,y in zip(max_cols,max_vals)]

In [76]:
all_models_df.to_csv('model_comparison_21_10_25.csv')

In [73]:
len('_median_real')

12